<a href="https://colab.research.google.com/github/evansi938/Project-33-Capstone/blob/main/Graph_Attention_Network_Attachment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install angr networkx torch-geometric

In [ ]:
%pip install --upgrade angr

In [ ]:
! git clone https://github.com/iosifache/DikeDataset.git

Cloning into 'DikeDataset'...
remote: Enumerating objects: 11989, done.
remote: Counting objects: 100% (11989/11989), done.
remote: Compressing objects: 100% (9243/9243), done.
remote: Total 11989 (delta 2749), reused 11981 (delta 2741), pack-reused 0 (from 0)
Receiving objects: 100% (11989/11989), 2.38 GiB | 22.59 MiB/s, done.
Resolving deltas: 100% (2749/2749), done.
Updating files: 100% (11936/11936), done.


I am still having some difficulties with the below code section:

In [ ]:
import os
import glob
import torch
import angr
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

def extract_cfg_data(binary_path, label, timeout_seconds=45, max_nodes=100):
  # Extract one binary from the DikeDataset then build the CFG/return object
  try:
    proj = angr.Project(binary_path, auto_load_libs=False)
    cfg = proj.analyses.CFGFast()

    if len(cfg.graph.nodes) == 0:
      print(f"Skipping {os.path.basename(binary_path)}: graph exceeds {max_nodes} nodes.")
      return None

    g = nx.DiGraph()
    for node in cfg.graph.nodes():
      sz = node.size if node.size is not None else 0
      insn_cnt = len(node.instruction_addrs) if hasattr(node, 'instruction_addrs') else 0
      g.add_node(node.addr, size = sz, insn_count = insn_cnt)

    for source, destination, data in cfg.graph.edges(data=True):
      g.add_edge(source.addr, destination.addr)

    mapping = {addr: i for i, addr in enumerate(g.nodes())}

    features = []
    for n in g.nodes():
      feat = [
          float(g.nodes[n].get('size', 0)),
          float(g.nodes[n].get('insn_count', 0)),
          float(g.in_degree(n)),
          float(g.out_degree(n))
      ]
      features.append(feat)

    x = torch.tensor(features, dtype=torch.float)

    src = [mapping[u] for u, v in g.edges()]
    dst = [mapping[v] for u, v in g.edges()]
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=torch.tensor([label], dtype=torch.long))
    return data

  except Exception as e:
    print(f"Failed processing {os.path.basename(binary_path)}: e")
    return None

ERROR:angr.state_plugins.unicorn_engine:failed loading "unicornlib.so", unicorn support disabled ('NoneType' object has no attribute 'unicorn_py3')


In [ ]:
benign_path = "/content/DikeDataset/files/benign"; # I am waiting on the dikedataset to fully download.
malware_path = "/content/DikeDataset/files/malware";

benign_files = glob.glob(os.path.join(benign_path, "*"))[:30]
malware_files = glob.glob(os.path.join(malware_path, "*"))[:30]

dataset = []

print("Process benign files:")
for path in benign_files:
  data = extract_cfg_data(path, label = 0)
  if data is not None:
    dataset.append(data)

print("Process malware files:")
for path in malware_files:
  data = extract_cfg_data(path, label=1)
  if data is not None:
    dataset.append(data)

print(f"Number of graph samples: {len(dataset)}")

loader = DataLoader(dataset, batch_size=4, shuffle=True)


Process benign files:


Skipping 66a2604f0910eca2751e4d5eee487791cb0fcfdf00ba457a4aa4be4b32f6d8ce.exe: graph exceeds 100 nodes.
Failed processing e4e48ec4f35ab5a21f791b014d1c626d6758906bf6ecff845ae3a6be90a97f1c.ole: e


ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty amd64g_dirtyhelper_IN.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty amd64g_dirtyhelper_OUT.


Failed processing 26ddd2790fff669f1fdcd301f777ec8975b88a2e8d6d023a1576cebbe81f76e9.ole: e
Failed processing 37df607966b7219faf58bb94e931b455b27e86c52c54878307c9a4c564067cf4.exe: e
Failed processing 755c00b1e667605746edc0eb826412dca4f317fd290728b5c71e6a7c12f537ed.ole: e
Process malware files:


ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_IN.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_OUT.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_FRSTOR.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_FLDENV.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_FINIT.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_FSTENV.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_RDMSR.
ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_FSAVE.


Failed processing f198ab670a8b0340f9f9564522fcd3ec55bf271c8497766810c9ddfde0105648.ole: e
Failed processing 259cb6d2f24b11cef194a139e7b63eb217a1090ebeb963b074ea606726df7e4a.ole: e
Failed processing 5d9edc02f04ddf9fb8eeb524fba385c85303fdb91b154a912a32edc47763657c.ole: e


ERROR:angr.analyses.fcp.fcp.SimEngineFCPVEX:Unsupported Dirty x86g_dirtyhelper_write_cr0.


Failed processing 3b126dca8990f6cbe0cf6158212a6e62544b8efc0b45226c7f317163d70b60fc.ole: e
Number of graph samples: 51
